# XGBoost Model Training & Evaluation
Train and evaluate all 5 prediction models.

In [ ]:
import sys
sys.path.insert(0, '../backend')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ml.features import build_feature_vector
from ml.model import train_all_models, load_models, predict, TARGET_GROUPS

with open('../backend/data/seed_perfumes.json') as f:
    perfumes = json.load(f)

print(f'Training on {len(perfumes)} perfumes')

In [ ]:
# Train all models
train_all_models(perfumes)
models = load_models()
print(f'Loaded {len(models)} models: {list(models.keys())}')

In [ ]:
# Test prediction on Dior Sauvage EDT
sauvage = next(p for p in perfumes if p['name'] == 'Sauvage' and p['brand'] == 'Dior' and p['concentration'] == 'EDT')
result = predict(sauvage, models)

print('=== Dior Sauvage EDT Predictions ===')
for k, v in result.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.2f}')
    else:
        print(f'  {k}: {v}')

In [ ]:
# Compare top 5 perfumes by blind buy score
results = []
for p in perfumes:
    r = predict(p, models)
    results.append({
        'perfume': f"{p['brand']} {p['name']} {p['concentration']}",
        'blind_buy': r['blind_buy_score'],
        'longevity_h': r['longevity_hours'],
        'sillage': r['sillage_score'],
        'versatility': r['versatility_score'],
    })

results_df = pd.DataFrame(results).sort_values('blind_buy', ascending=False)
print('Top 10 by blind buy score:')
results_df.head(10)

In [ ]:
# Heatmap of performance predictions across all seed perfumes
import seaborn as sns

score_cols = ['blind_buy', 'longevity_h', 'sillage', 'versatility']
heatmap_df = results_df.set_index('perfume')[score_cols]

plt.figure(figsize=(10, 12))
sns.heatmap(heatmap_df, annot=True, fmt='.1f', cmap='YlOrRd', vmin=0, vmax=10)
plt.title('Prediction Scores — All Seed Perfumes')
plt.tight_layout()
plt.show()